# Module 8: SQL & Advanced Operations

## Learning Objectives
By the end of this module, you will understand:
- Creating temporary views
- Writing SQL queries
- SQL expressions in PySpark
- Conditional expressions (CASE WHEN)
- Combining SQL and DataFrame API

---

## 1. Temporary Views

Convert DataFrames to SQL tables for querying.

# Create sample data
data = [
    (1, "Alice", "Engineering", 70000),
    (2, "Bob", "Sales", 85000),
    (3, "Charlie", "Engineering", 75000),
    (4, "Diana", "Sales", 95000),
    (5, "Eve", "Engineering", 72000)
]
df = spark.createDataFrame(data, ["emp_id", "name", "department", "salary"])

# Create temporary view
df.createOrReplaceTempView("employees")

print("DataFrame registered as 'employees' view")
print("\nQuerying with SQL:")
result = spark.sql("SELECT * FROM employees")
result.show()

**Output:**
```
DataFrame registered as 'employees' view

Querying with SQL:
+------+-------+----------+------+
|emp_id|   name|department|salary|
+------+-------+----------+------+
|     1|  Alice|Engineering| 70000|
|     2|    Bob|     Sales| 85000|
|     3|Charlie|Engineering| 75000|
|     4|  Diana|     Sales| 95000|
|     5|    Eve|Engineering| 72000|
+------+-------+----------+------+
```

# Global temporary view (accessible across sessions)
df.createOrReplaceGlobalTempView("employees_global")

# Must reference as global_temp.employees_global
result = spark.sql("SELECT * FROM global_temp.employees_global WHERE salary > 80000")
print("Global temp view - high earners:")
result.show()

**Output:**
```
Global temp view - high earners:
+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     2|  Bob|     Sales| 85000|
|     4|Diana|     Sales| 95000|
+------+-----+----------+------+
```

---

## 2. SQL Queries

Full SQL capability through Spark.

# SELECT with WHERE
print("Engineers earning more than 70000:")
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE department = 'Engineering' AND salary > 70000
    ORDER BY salary DESC
""").show()

**Output:**
```
Engineers earning more than 70000:
+-------+------+
|   name|salary|
+-------+------+
|Charlie| 75000|
|    Eve| 72000|
+-------+------+
```

# GROUP BY and aggregation
print("\nSalary statistics by department:")
spark.sql("""
    SELECT 
        department,
        COUNT(*) as employee_count,
        AVG(salary) as avg_salary,
        MAX(salary) as max_salary,
        MIN(salary) as min_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

**Output:**
```
Salary statistics by department:
+----------+--------------+-----------+-----------+-----------+
|department|employee_count|avg_salary |max_salary |min_salary |
+----------+--------------+-----------+-----------+-----------+
|   Sales  |             2|    90000.0|     95000 |     85000 |
|Engineering|             3|    72333.33|     75000 |     70000 |
+----------+--------------+-----------+-----------+-----------+
```

# HAVING clause
print("\nDepartments with average salary > 75000:")
spark.sql("""
    SELECT 
        department,
        AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 75000
""").show()

**Output:**
```
Departments with average salary > 75000:
+----------+-----------+
|department|avg_salary |
+----------+-----------+
|   Sales  |   90000.0 |
+----------+-----------+
```

---

## 3. Conditional Expressions (CASE WHEN)

from pyspark.sql.functions import when, col, case

# Method 1: Using when/otherwise (DataFrame API)
print("Salary brackets (DataFrame API):")
df.withColumn(
    "salary_bracket",
    when(col("salary") >= 80000, "High")
    .when(col("salary") >= 70000, "Medium")
    .otherwise("Low")
).select("name", "salary", "salary_bracket").show()

**Output:**
```
Salary brackets (DataFrame API):
+-------+------+-----------+
|   name|salary|salary_bracket|
+-------+------+-----------+
|  Alice| 70000|    Medium |
|    Bob| 85000|     High  |
|Charlie| 75000|    Medium |
|  Diana| 95000|     High  |
|    Eve| 72000|    Medium |
+-------+------+-----------+
```

# Method 2: Using CASE in SQL
print("\nSalary brackets (SQL):")
spark.sql("""
    SELECT 
        name,
        salary,
        CASE 
            WHEN salary >= 80000 THEN 'High'
            WHEN salary >= 70000 THEN 'Medium'
            ELSE 'Low'
        END as salary_bracket
    FROM employees
    ORDER BY salary DESC
""").show()

**Output:**
```
Salary brackets (SQL):
+-------+------+-----------+
|   name|salary|salary_bracket|
+-------+------+-----------+
|  Diana| 95000|     High  |
|    Bob| 85000|     High  |
|Charlie| 75000|    Medium |
|    Eve| 72000|    Medium |
|  Alice| 70000|    Medium |
+-------+------+-----------+
```

---

## 4. Combining SQL and DataFrame API

Mix SQL queries with DataFrame transformations.

# Start with SQL, then transform with DataFrame API
high_earners = spark.sql("SELECT * FROM employees WHERE salary > 75000")

print("SQL result, then DataFrame transformation:")
high_earners.select("name", "department", "salary").show()

# Further transform the SQL result
result = high_earners.withColumn(
    "bonus",
    col("salary") * 0.1
).select("name", "salary", "bonus")

print("\nWith added bonus column:")
result.show()

**Output:**
```
SQL result, then DataFrame transformation:
+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|    Bob|     Sales| 85000|
|Charlie|Engineering| 75000|
|  Diana|     Sales| 95000|
+-------+----------+------+

With added bonus column:
+-------+------+-----+
|   name|salary|bonus|
+-------+------+-----+
|    Bob| 85000| 8500|
|Charlie| 75000| 7500|
|  Diana| 95000| 9500|
+-------+------+-----+
```

# DataFrame to SQL and back
df_computed = df.withColumn(
    "salary_category",
    when(col("salary") >= 80000, "Senior")
    .when(col("salary") >= 70000, "Mid")
    .otherwise("Junior")
)

# Register it as a view
df_computed.createOrReplaceTempView("employees_ranked")

# Now query it with SQL
print("\nQuery the computed DataFrame with SQL:")
spark.sql("""
    SELECT 
        salary_category,
        COUNT(*) as count,
        AVG(salary) as avg_salary
    FROM employees_ranked
    GROUP BY salary_category
    ORDER BY avg_salary DESC
""").show()

**Output:**
```
Query the computed DataFrame with SQL:
+-----------+-----+-----------+
|salary_cat |count|avg_salary |
+-----------+-----+-----------+
|   Senior  |    2|   90000.0 |
|    Mid    |    3|   72333.33|
+-----------+-----+-----------+
```

---

## 5. SQL Expressions

Use `expr()` to write SQL expressions in DataFrame API.

from pyspark.sql.functions import expr

# Use SQL expressions in select
print("Using expr() for complex calculations:")
df.select(
    "name",
    expr("salary * 0.1 as bonus"),
    expr("salary + (salary * 0.1) as total_comp"),
    expr("CASE WHEN salary >= 80000 THEN 'High' ELSE 'Low' END as level")
).show()

**Output:**
```
Using expr() for complex calculations:
+-------+------+----------+-----+
|   name| bonus|total_comp|level|
+-------+------+----------+-----+
|  Alice|  7000|     77000|  Low|
|    Bob|  8500|     93500| High|
|Charlie|  7500|     82500|  Low|
|  Diana|  9500|    104500| High|
|    Eve|  7200|     79200|  Low|
+-------+------+----------+-----+
```

---

## 6. Mini Challenges

### Challenge 1: Temporary Views
1. Create a DataFrame
2. Register as temp view
3. Write SQL SELECT query
4. Write SQL with WHERE clause
5. Write SQL with GROUP BY

### Challenge 2: CASE Statements
1. Create salary bracket column using CASE WHEN
2. Query with GROUP BY on bracket
3. Calculate aggregate stats by bracket

### Challenge 3: Mixed Approach
1. Create DataFrame
2. Add computed columns with expr()
3. Register as view
4. Query with SQL
5. Transform results with DataFrame API

---

## Key Takeaways

✅ **createOrReplaceTempView()** - Register DataFrame as table

✅ **spark.sql()** - Execute SQL queries

✅ **CASE WHEN** - Conditional logic in SQL

✅ **when().otherwise()** - Conditional in DataFrame API

✅ **expr()** - Write SQL expressions in DataFrame API

✅ **Mix SQL + DataFrame API** - Use both approaches together

---

## You've Completed the Full PySpark Tutorial!

### What You've Learned:
✅ Module 1: Databricks fundamentals, DBFS, SparkSession, DataFrame creation

✅ Module 2: Reading/writing CSV, JSON, Parquet, Delta Lake

✅ Module 3: Exploring data, selecting columns, renaming, statistics

✅ Module 4: Transformations - casting, string/math/date functions

✅ Module 5: Data cleaning - filtering, nulls, deduplication

✅ Module 6: Aggregations and grouping

✅ Module 7: Sorting and joining DataFrames

✅ Module 8: SQL queries and advanced operations

### Next Steps:
- Practice with real datasets
- Explore Databricks MLlib for machine learning
- Learn about performance optimization and partitioning
- Build production pipelines with Delta Lake